In [ ]:
from pathlib import Path
from typing import List
import lightgbm as lgb
import mlflow
import pandas as pd

import energycast.evaluation.metrics as metrics
import energycast.utils.data_utils as data_utils
import energycast.utils.model_artifacts as model_artifacts
from energycast.models import model_utils

In [ ]:
conf = model_utils.load_model_config("lgbm_per_customer_24")
train = data_utils.load_parquet("PROCESSED", conf.train_set)
dev = data_utils.load_parquet("PROCESSED", conf.dev_set)
test = data_utils.load_parquet("PROCESSED", conf.test_set)

'default_24_dev'

In [ ]:
def per_customer_data(
    train: pd.DataFrame,
    dev: pd.DataFrame,
    test: pd.DataFrame,
    features: List[str],
    target: str,
):

    customer_data = {
        customer: {
            "X_train": train.loc[train["object_id"] == customer, features],
            "y_train": train.loc[train["object_id"] == customer, target],
            "X_dev": dev.loc[dev["object_id"] == customer, features],
            "y_dev": dev.loc[dev["object_id"] == customer, target],
            "X_test": test.loc[test["object_id"] == customer, features],
            "y_test": test.loc[test["object_id"] == customer, target],
            "y_pred": 0,
        }
        for customer in train["object_id"].unique()
    }
    return customer_data


def train_models_per_customer(customer_data, config):
    customer_models = {}
    for customer, data in customer_data.items():
        model = lgb.LGBMRegressor(
            objective=config["objective"],
            boosting_type=config["boosting_type"],
            metric=config["metric"],
            random_state=config["random_state"],
            learning_rate=config["learning_rate"],
            num_leaves=config["num_leaves"],
            max_depth=config["max_depth"],
            min_data_in_leaf=config["min_data_in_leaf"],
            n_estimators=config["n_estimators"],
            lambda_l1=config["lambda_l1"],
            lambda_l2=config["lambda_l2"],
        )
        model.fit(
            data["X_train"],
            data["y_train"],
            eval_set=[(data["X_dev"], data["y_dev"])],
            eval_metric="mae",
        )
        customer_models[customer] = model

    return customer_models


def predict_per_customer(customer_models, customer_data):
    for customer, model in customer_models.items():
        customer_data[customer]["y_pred"] = model.predict(
            customer_data[customer]["X_test"]
        )
        prediction = [
            pd.DataFrame(
                {
                    "object_id": customer,
                    "y_true": customer_data[customer]["y_test"]
                    .to_numpy()
                    .reshape(-1),
                    "y_pred": customer_data[customer]["y_pred"],
                }
            )
            for customer in customer_data.keys()
        ]

    return pd.concat(prediction)

In [ ]:
customer_data = per_customer_data(
    train, dev, test, conf.features, conf.target_col
)
customer_models = train_models_per_customer(customer_data, conf.model_params)
prediction = predict_per_customer(customer_models, customer_data)

In [73]:
prediction = predict_per_customer(customer_models, customer_data)

[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] lambda_l2 is set=0.5, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.5
[LightGBM] [Warning] lambda_l1 is set=0.5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.5
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] lambda_l2 is set=0.5, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.5
[LightGBM] [Warning] lambda_l1 is set=0.5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.5
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] lambda_l2 is set=0.5, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.5
[LightGBM] [Warning] lambda_l1 is set=0.5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.5
[LightG